# 1.04 微积分与自动求导

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/01_prerequisites/1.04_calculus_autograd.ipynb)


## 为什么要学

"训练一个模型"本质上就是一句话：**沿着损失函数下降最快的方向，一点点调整参数**。"下降最快的方向"就是梯度的反方向。而 PyTorch 的 `loss.backward()` 能自动算出几十亿个参数的梯度，靠的是**链式法则 + 计算图**。

本课会**从零实现一个迷你自动求导引擎**（参考 Karpathy 的 [micrograd](https://github.com/karpathy/micrograd)）。写完它，`backward()` 对你来说就不再是黑箱。

---

## 1. 导数：变化率

$$
f'(x) = \lim_{h\to 0}\frac{f(x+h) - f(x)}{h}
$$

导数告诉我们：$x$ 增加一点点，$f$ 会变化多少。可以用一个很小的 $h$ 数值近似。**这个"数值梯度"是检查你手写梯度是否正确的标准方法**。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

f = lambda x: x ** 3 - 2 * x
df = lambda x: 3 * x ** 2 - 2          # 手算的导数
x0, h = 1.5, 1e-5
numeric = (f(x0 + h) - f(x0 - h)) / (2 * h)    # 中心差分，比单边差分更准
print("数值导数:", numeric, " 解析导数:", df(x0))

## 2. 偏导数与梯度

多元函数 $f(\mathbf{x})$ 对每个分量分别求导，得到**梯度**向量：

$$
\nabla f(\mathbf{x}) = \left[\frac{\partial f}{\partial x_1}, \dots, \frac{\partial f}{\partial x_n}\right]^\top
$$

**梯度指向函数值上升最快的方向**。所以梯度下降就是：

$$
\mathbf{x} \leftarrow \mathbf{x} - \eta\,\nabla f(\mathbf{x})
$$

$\eta$ 是学习率（步长）。下面在一个二维"碗"上可视化梯度下降，并观察学习率太大时会发生什么：

In [ ]:
def f2(p):   # 一个椭圆形的碗：x 方向平缓，y 方向陡峭
    return 0.5 * p[0] ** 2 + 5 * p[1] ** 2
def grad_f2(p):
    return np.array([p[0], 10 * p[1]])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
xs, ys = np.meshgrid(np.linspace(-4, 4, 100), np.linspace(-2, 2, 100))
for ax, lr in zip(axes, [0.02, 0.15, 0.21]):
    p = np.array([-3.5, 1.5]); path = [p.copy()]
    for _ in range(30):
        p = p - lr * grad_f2(p)
        path.append(p.copy())
    path = np.array(path)
    ax.contour(xs, ys, f2([xs, ys]), levels=20, cmap="Blues")
    ax.plot(path[:, 0], path[:, 1], "r.-")
    ax.set_title(f"lr={lr}  final f={f2(p):.3g}")
plt.tight_layout(); plt.show()

可以看到：学习率太小走得慢；在 y 方向（陡峭方向）学习率超过 $2/10=0.2$ 就会**来回震荡发散**。这就是 Adam 等"自适应学习率"优化器要解决的问题（1.08 课讲）。

## 3. 链式法则与计算图

如果 $y = f(u)$、$u = g(x)$，那么

$$
\frac{dy}{dx} = \frac{dy}{du}\cdot\frac{du}{dx}
$$

神经网络就是很多函数的复合。把计算过程画成一张**计算图**：每个节点是一个中间结果，反向传播就是**从输出往回走，每个节点把"上游传来的梯度"乘以"本地导数"，再传给它的输入**。

以 $L = (wx + b - y)^2$ 为例（一个样本的线性回归损失）：

```text
 前向:  w,x ──(*)──> z1 ──(+b)──> z2 ──(-y)──> z3 ──(^2)──> L
 反向:  dL/dL = 1
        dL/dz3 = 2*z3
        dL/dz2 = dL/dz3 * 1
        dL/dz1 = dL/dz2 * 1      dL/db = dL/dz2 * 1
        dL/dw  = dL/dz1 * x      dL/dx = dL/dz1 * w
```

**关键点**：如果一个变量被用了多次，它的梯度是各条路径传回来的梯度**之和**（多元链式法则）。

---

## 4. 从零实现自动求导：`Value` 类

我们实现一个只支持标量的 `Value`：它记住自己的数值 `data`、梯度 `grad`、以及"它是由哪些节点、通过什么运算得到的"。

In [ ]:
class Value:
    """一个标量节点。参考 micrograd（Karpathy）。"""

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0                  # dL/d(self)，反向传播时累加
        self._backward = lambda: None    # 把 self.grad 传给子节点的函数
        self._prev = set(_children)      # 产生这个节点的输入节点
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            # 加法的本地导数都是 1：上游梯度原样传下去（注意是 += 累加）
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        raise NotImplementedError("练习 1 中实现")

    def __pow__(self, k):
        assert isinstance(k, (int, float))
        out = Value(self.data ** k, (self,), f"**{k}")
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        raise NotImplementedError("练习 1 中实现")

    # 下面这些让 Value 支持 a - b、-a、2 * a 等写法
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other

    def backward(self):
        # 1) 拓扑排序：保证处理一个节点前，所有"用到它"的节点都已经处理完
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        # 2) 从输出开始，反向依次调用每个节点的 _backward
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

### 练习 1：实现乘法和 tanh 的反向传播

- 乘法 $out = a\cdot b$：$\frac{\partial out}{\partial a} = b$，$\frac{\partial out}{\partial b} = a$
- $out = \tanh(a)$：$\frac{d\,out}{da} = 1 - \tanh^2(a)$

In [ ]:
def _mul(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), "*")
    def _backward():
        # TODO: 把 out.grad 乘以本地导数，累加到 self.grad 与 other.grad
        raise NotImplementedError
    out._backward = _backward
    return out

def _tanh(self):
    t = np.tanh(self.data)
    out = Value(t, (self,), "tanh")
    def _backward():
        # TODO
        raise NotImplementedError
    out._backward = _backward
    return out

Value.__mul__ = _mul
Value.tanh = _tanh

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def _mul(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), "*")
    def _backward():
        self.grad += other.data * out.grad    # d(a*b)/da = b
        other.grad += self.data * out.grad    # d(a*b)/db = a
    out._backward = _backward
    return out

def _tanh(self):
    t = np.tanh(self.data)
    out = Value(t, (self,), "tanh")
    def _backward():
        self.grad += (1 - t ** 2) * out.grad
    out._backward = _backward
    return out

Value.__mul__ = _mul
Value.tanh = _tanh

In [ ]:
# ✅ 自测：和 PyTorch 的 autograd 对比
a, b, c = Value(2.0), Value(-3.0), Value(0.5)
L = ((a * b + c).tanh() * a + b ** 2) * c        # 一个随便写的复合函数，注意 a、b 都被用了两次
L.backward()

ta, tb, tc = (torch.tensor(v, requires_grad=True) for v in (2.0, -3.0, 0.5))
TL = (torch.tanh(ta * tb + tc) * ta + tb ** 2) * tc
TL.backward()
for name, mine, ref in [("a", a, ta), ("b", b, tb), ("c", c, tc)]:
    assert abs(mine.grad - ref.grad.item()) < 1e-6, f"{name} 的梯度不对: {mine.grad} vs {ref.grad.item()}"
print("通过 ✅ 你的自动求导与 PyTorch 完全一致:", a, b, c)

### 用 `Value` 训练一个神经元

In [ ]:
import random
random.seed(0)
# 数据：y = tanh(2x - 1)
xs = [random.uniform(-2, 2) for _ in range(20)]
ys = [np.tanh(2 * x - 1) for x in xs]
w, b = Value(0.1), Value(0.0)
for step in range(200):
    loss = sum(((w * x + b).tanh() - y) ** 2 for x, y in zip(xs, ys)) * (1 / len(xs))
    w.grad = b.grad = 0.0          # ⚠️ 梯度是累加的，每步之前要清零
    loss.backward()
    w.data -= 0.5 * w.grad         # 梯度下降
    b.data -= 0.5 * b.grad
    if step % 50 == 0:
        print(f"step {step:3d} loss={loss.data:.5f} w={w.data:.3f} b={b.data:.3f}")
print("学到的 w, b ≈", round(w.data, 3), round(b.data, 3), "（真实值 2, -1）")

---

## 5. PyTorch 的 autograd

PyTorch 做的事和我们的 `Value` 一模一样，只不过节点是**张量**，并且支持几百种运算。

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)   # 告诉 PyTorch：要对 x 求梯度
y = (x ** 2).sum()      # y = x1^2 + x2^2 + x3^2
y.backward()            # 反向传播
print(x.grad)           # dy/dx = 2x = [2, 4, 6]

# ⚠️ 梯度会累加：再 backward 一次会变成 [4, 8, 12]
y = (x ** 2).sum(); y.backward(); print(x.grad)
x.grad.zero_()          # 训练循环里的 optimizer.zero_grad() 就是做这个

### 三个必须掌握的工具：`no_grad`、`detach`、`requires_grad_(False)`

| 写法 | 作用 | 典型场景 |
|---|---|---|
| `with torch.no_grad():` | 这段代码不建计算图 | 推理、评估、更新目标网络 |
| `x.detach()` | 得到一个"切断梯度"的副本 | RLT 中冻结 VLA 的特征；DQN 的 TD 目标；GAN |
| `p.requires_grad_(False)` | 冻结参数 | 只训练动作头、冻结视觉编码器 |

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
feat = w * 3                    # 假设这是 VLA 的特征
head_out = feat.detach() * 5    # RLT：RL 头只用特征，不把梯度传回 VLA
loss = head_out ** 2
print("loss 是否连着 w 的计算图:", loss.requires_grad)   # False：梯度传不到 w

with torch.no_grad():
    z = w * 100
print("no_grad 下的结果 requires_grad:", z.requires_grad)

> 在本项目的 RLT 实现里（第 8 章），`RLTTokenTransformer.reconstruct` 第一行就是 `prefix_embs.detach()`。这样 RL token 的训练**不会改动** VLA 本身。这正是"冻结 VLA、只训练小网络"的实现方式。

### 练习 2：数值梯度检查

写一个函数，用中心差分计算任意函数 `f`（输入张量 → 标量）在 `x` 处的数值梯度。用它验证 autograd 的结果。**以后你自己写复杂的损失函数时，这是排查梯度 bug 的利器**。

In [ ]:
def numerical_grad(f, x, h=1e-4):
    """f: 张量 -> 标量张量；x: 任意形状的 float64 张量。返回与 x 同形状的数值梯度。"""
    grad = torch.zeros_like(x)
    # TODO: 遍历 x 的每个元素（用 x.view(-1) 和 grad.view(-1)），分别 +h、-h 计算中心差分
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def numerical_grad(f, x, h=1e-4):
    """f: 张量 -> 标量张量；x: 任意形状的 float64 张量。返回与 x 同形状的数值梯度。"""
    grad = torch.zeros_like(x)
    xf, gf = x.view(-1), grad.view(-1)       # 展平的"视图"，修改它们就是修改原张量
    for i in range(xf.numel()):
        old = xf[i].item()
        xf[i] = old + h; fp = f(x).item()
        xf[i] = old - h; fm = f(x).item()
        xf[i] = old                           # 记得恢复原值
        gf[i] = (fp - fm) / (2 * h)
    return grad

In [ ]:
# ✅ 自测：检查 softmax 交叉熵的梯度
torch.manual_seed(0)
logits = torch.randn(3, 5, dtype=torch.float64)
target = torch.tensor([1, 0, 4])
f = lambda z: torch.nn.functional.cross_entropy(z, target)
z = logits.clone().requires_grad_(True); f(z).backward()
num = numerical_grad(f, logits.clone())
print("最大误差:", (num - z.grad).abs().max().item())
assert torch.allclose(num, z.grad, atol=1e-6)
print("通过 ✅")

---

## 小结

- 梯度 = 上升最快的方向；梯度下降 = 沿反方向走一小步。学习率太大会发散。
- 反向传播 = 链式法则 + 计算图 + 拓扑排序；被多次使用的变量，梯度要**累加**。
- PyTorch 中：`backward()` 前要清零梯度；推理用 `no_grad`；冻结特征用 `detach`。

**思考题**：我们的 `Value.backward()` 为什么要先拓扑排序？如果直接用递归"先处理自己再处理子节点"，在 `a` 被用了两次的情况下会出什么错？